# Week 3 - BSM Chooser Option Model Validation

## Objective

Provide a separate, auditable validation record for the Week 3 chooser option model. This notebook reads the outputs generated by `Week3.ipynb` and checks:

1. the comparison with the paper's reported Table 3 paths and payoffs;
2. the Monte Carlo comparison with the closed-form BSM chooser price; and
3. all analytical validation tests.

Run `Week3.ipynb` first so that the files in `model_results/` are current.


## 1. Load Validation Outputs

In [1]:
from pathlib import Path

import pandas as pd

RESULTS_DIR = Path("./model_results")
required_files = {
    "Paper comparison": RESULTS_DIR / "paper_table3_comparison.csv",
    "Monte Carlo validation": RESULTS_DIR / "monte_carlo_validation.csv",
    "Validation summary": RESULTS_DIR / "validation_summary.csv",
    "Parameter configuration": RESULTS_DIR / "parameter_configuration.csv",
}

missing_files = [str(path) for path in required_files.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Run Week3.ipynb first. Missing validation files: " + ", ".join(missing_files)
    )

parameter_configuration = pd.read_csv(required_files["Parameter configuration"])
paper_comparison = pd.read_csv(required_files["Paper comparison"])
monte_carlo_validation = pd.read_csv(required_files["Monte Carlo validation"])
validation_summary = pd.read_csv(required_files["Validation summary"])

pd.DataFrame(
    {
        "Output": list(required_files),
        "File": [str(path.resolve()) for path in required_files.values()],
    }
)


                    Output                                               File
0         Paper comparison  G:\JPM-Chooser Option Pricing\Week 3\model_res...
1   Monte Carlo validation  G:\JPM-Chooser Option Pricing\Week 3\model_res...
2       Validation summary  G:\JPM-Chooser Option Pricing\Week 3\model_res...
3  Parameter configuration  G:\JPM-Chooser Option Pricing\Week 3\model_res...

## 2. Confirm the Required Parameter Configuration

In [2]:
required_paper_values = {
    "K": 150.0,
    "T1": 0.5,
    "T2": 1.0,
}

paper_row = parameter_configuration.loc[
    parameter_configuration["Configuration"].eq("Paper replication")
].iloc[0]

parameter_checks = pd.DataFrame(
    {
        "Parameter": list(required_paper_values),
        "Required": list(required_paper_values.values()),
        "Observed": [paper_row[name] for name in required_paper_values],
    }
)
parameter_checks["Pass"] = (
    parameter_checks["Required"] - parameter_checks["Observed"]
).abs() < 1e-12

parameter_checks


  Parameter  Required  Observed  Pass
0         K     150.0     150.0  True
1        T1       0.5       0.5  True
2        T2       1.0       1.0  True

## 3. Paper Table 3 Comparison

The original paper does not provide its random seed or shocks, so the same stock-price paths cannot be regenerated honestly. The published choices and terminal payoff arithmetic can, however, be checked exactly.


In [3]:
paper_comparison_summary = pd.DataFrame(
    {
        "Metric": [
            "Reported rows checked",
            "Choice mismatches",
            "Maximum payoff absolute error",
        ],
        "Value": [
            len(paper_comparison),
            (paper_comparison["Paper_Choice"] != paper_comparison["Recalculated_Choice"]).sum(),
            paper_comparison["Payoff_Absolute_Error"].max(),
        ],
    }
)

paper_comparison_summary


                          Metric         Value
0          Reported rows checked  1.000000e+01
1              Choice mismatches  0.000000e+00
2  Maximum payoff absolute error  1.421085e-14

In [4]:
paper_comparison

   First_6M_Stock Paper_Choice  ...  Recalculated_Payoff  Payoff_Absolute_Error
0          118.33          PUT  ...                33.23           7.105427e-15
1          222.63         CALL  ...                42.89           1.421085e-14
2          186.53         CALL  ...                42.94           0.000000e+00
3          164.08         CALL  ...                 0.00           0.000000e+00
4          159.09         CALL  ...                 0.00           0.000000e+00
5          186.73         CALL  ...                 0.00           0.000000e+00
6          106.61          PUT  ...                59.48           7.105427e-15
7          163.06         CALL  ...                29.61           1.421085e-14
8          129.26          PUT  ...                 5.18           7.105427e-15
9          115.41          PUT  ...                13.50           0.000000e+00

[10 rows x 7 columns]

## 4. Closed-Form vs Monte Carlo Comparison

In [5]:
monte_carlo_validation

    Paths      Seed  ...  Absolute Difference  Within 95% CI
0  250000  20260806  ...             0.050431           True

[1 rows x 10 columns]

## 5. Complete Validation Summary

In [6]:
validation_summary


                                     Validation Test      Observed  Pass
0                                    Put-call parity  0.000000e+00  True
1                                    Chooser >= call  1.044093e+01  True
2                                     Chooser >= put  1.375685e+01  True
3                              Chooser <= call + put  4.932143e+00  True
4        Paper Table 3 choices follow the paper rule  1.000000e+00  True
5          Paper Table 3 payoffs recalculate exactly  1.421085e-14  True
6  Monte Carlo is within three standard errors of...  8.227574e-01  True

In [7]:
assert parameter_checks["Pass"].all(), "Required parameter configuration failed."
assert validation_summary["Pass"].all(), "At least one model validation test failed."
assert (paper_comparison["Payoff_Absolute_Error"] < 1e-10).all(), "Paper payoff comparison failed."
assert bool(monte_carlo_validation.loc[0, "Within 95% CI"]), "Closed-form price is outside the Monte Carlo 95% CI."

print("All Week 3 parameter, paper-comparison, analytical, and Monte Carlo checks passed.")


All Week 3 parameter, paper-comparison, analytical, and Monte Carlo checks passed.


# Validation Conclusion

The Week 3 model passes all configured checks. The paper's reported payoff arithmetic is reproduced exactly, and the analytical chooser price is consistent with the independently simulated Monte Carlo estimate. The paper's undisclosed random seed remains a documented reproducibility limitation.
